# IFC Data Analysis

In this file we shall attempt to load and edit data present in an IFC file. This includes metadata and geometry (if applicable). Let's start with importing a new library - `ifcopenshell`.

In [1]:
import ifcopenshell

This library assists with data cleaning, and management section of IFC data (relates to the metadata behind the objects of the scene).

Loading the model entails calling the following code. Here, we are working with the MEP model from our `bimsixty5` model.

In [2]:
model = ifcopenshell.open('data/O-S1-INS-Plumbing - Sanitair.ifc')

From the docs, it seems prudent to verify this import went accurately. Let's verify the IFC schema of this file.

In [3]:
print(model.schema)

IFC2X3


This is an older version of the IFC schema, but looks like we got the model loading in properly. Let's get the metadata associated with the first object in the scene.

In [4]:
print(model.by_id(1))

#1=IfcOrganization($,'Autodesk Revit 2018 (ENU)',$,$,$)


This seems to be the authoring information. Let's try to isolate a specific type of object using IFC schema.

In [5]:
element = model.by_type('IfcElement')[0]
print(element)

#8240=IfcBuildingElementProxy('00MeRCAavDEuoJ50DJZAxT',#42,'00_B4A_Referentiepunt:Kubus:1768758',$,'00_B4A_Referentiepunt:Kubus',#8239,#8234,'1768758',$)


We run into the first issue at hand, which is that the metadata is saved in dutch language. Looks like we'll need a tab of google translate open at all times. "Referentiepunt" means "Reference Point" and "Kubus" means "Cube". Likley, this is the default cube that gets loading in by default with the rest of the scene. We can ignore this one for now.

At this point, it seems useful to understand the basics of IFC schema.

## IFC Explained Briefly

IFC is a JSON style schema that is used to describe the built environment. The main concept of IFC objects is that they all fall into `classes`, each belonging to a hierarchy of sorts. An object in the scene can belong to multiple classes, depending on the metadata available to it.

Let's explore this concept. First, we shall select basic pipes in the model. Another useful resource we shall be using here is `bonsai`, a Blender add-in which allows you to view and edit IFC schema.

![Bonsai Overview](../img/bonsai-overview-2.png)

We can see that selecting a random pipe in the model shows us the specific name in the scene viewer on the left. We can peruse through the properties associated with this object in the panel on the right.

We can see very briefly in the panel on the right, the list of classes and types which we have available in the scene.

![IFC Classes Overview in Bonsai](../img/ifc-classes-bonsai-overview.png)

Let's see if we can narrow down our code results from earlier using this new class information.

In [8]:
flowSegment = model.by_type('IfcFlowSegment')[0]
print(flowSegment)

#263=IfcFlowSegment('1GU_X06Kf3OPsvt8DGONom',#42,'Pipe Types:Koper_Koper:1363938',$,'Pipe Types:Koper_Koper',#235,#257,'1363938')


Looks like we're able to isolate down to just the pipes.

Similarly, we can isolate down to the valves / fittings using the `class` called "IfcFlowController".

![IFC Flow Controller example](../img/ifcFlowController-example.png)

In [9]:
flowController = model.by_type('IfcFlowController')[0]
print(flowController)

#24536=IfcFlowController('1gpYmM12b6pxUJPBcN82UE',#42,'M_Meter_Water_MEPcontent_Unspecified:Generic Water Meter:1830925',$,'M_Meter_Water_MEPcontent_Unspecified:Generic Water Meter',#24535,#24530,'1830925')


Great, we now have a realiable way to isolate data from the model.

## Property Sets

Another concept within the IFC schema is that of property sets. This data structure saves direct metadata acssociated with a class, and an object can have mutliple of thses. Let's start by getting all the property sets and quantities associated with our pipes. This code is straight from the docs.

In [11]:
import ifcopenshell.util
import ifcopenshell.util.element
pset_flowSeg = ifcopenshell.util.element.get_psets(flowSegment)

print(pset_flowSeg)

{'Pset_DistributionFlowElementCommon': {'Reference': 'Koper_Koper', 'id': 274}, 'Pset_ElementShading': {'Roughness': 0.02, 'id': 280}, 'Pset_FlowSegmentDuctSegment': {'Length': 3914.99089320036, 'id': 283}, 'Pset_FlowSegmentPipeSegment': {'Length': 3914.99089320036, 'InvertElevation': 3813.2, 'id': 287}, 'Pset_ManufacturerTypeInformation': {'Manufacturer': 'Unspecified', 'id': 290}, 'Pset_ProductRequirements': {'Category': 'Pipes', 'id': 293}, 'Pset_QuantityTakeOff': {'Reference': 'Koper_Koper', 'id': 295}, 'M50_Leidingen - Zaaglijst': {'Family': 'Pipe Types: Koper_Koper', 'Buis type': 'Pipe Types: Koper_Koper', 'Diameter': 28.0, 'Lengte': 3914.99089320036, 'System Type': 'Piping System: W5310_Drinkwater', 'System Abbreviation': 'M531', 'id': 303}, 'M53_Sanitair - Zaaglijst': {'Family': 'Pipe Types: Koper_Koper', 'Buis type': 'Pipe Types: Koper_Koper', 'Diameter': 28.0, 'Lengte': 3914.99089320036, 'System Type': 'Piping System: W5310_Drinkwater', 'System Abbreviation': 'M531', 'id': 31

It might be easier to view this data in tabular format. Hence, we must convert this dictionary to a dataframe. We import our trusty libraries, pandas and numpy.

In [12]:
import pandas as pd
import numpy as np

## Extracting Tabular Data

Another tool available in the box is [IfcCSV](https://docs.ifcopenshell.org/ifccsv.html). This tool will let us export data out in tablular format. However, we do need to do some preprocessing to get it to this point.